# Synthetic EEG-like LSTM Sliding-Window Study — 3 Fixed Channels

All user-editable variables are centralized in the final control cell. Earlier cells only define imports, helpers, models, plotting utilities, and diagnostics.


In [19]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print("Google Colab is not available; skipping Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Functions


In [20]:
import math
import re
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

import random
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


def set_global_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds for deterministic/reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Disable backend auto-tuning / reduced-precision paths that can obscure exact repeatability.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.allow_tf32 = False

    # Enforce deterministic kernels instead of silently accepting nondeterministic alternatives.
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as exc:
        print(f"[WARN] Could not enable strict deterministic algorithms: {exc}")



## Configuration dataclass

Do not edit this cell for normal experiments. Edit the final control cell instead.


In [21]:
@dataclass
class Config:
    train_fs: int = 200
    test_fs: int = 200

    # Continuous-signal split by sample index.
    train_fraction: float = 0.75

    # Sliding-window settings used directly on the continuous sample stream.
    window_sec: float = 1.0
    stride_sec: float = 0.1

    # FFT band-power feature settings. For this synthetic beta signal, 18-22 Hz targets the carrier.
    bandpower_hz: tuple = (18.0, 22.0)

    # Single-channel loaded data. These fields are metadata placeholders only.
    idle_amp: tuple = (1.0,)
    active_amps: tuple = (1.0,)
    signal_noise_std: tuple = (0.0,)
    idle_beta_freq_hz: tuple = (20.0,)
    beta_freq_hz: tuple = (20.0,)

    batch_size: int = 64
    epochs: int = 12
    lr: float = 1e-3
    hidden_size: int = 4
    num_layers: int = 1


## Data loading and automatic labeling

This section loads a single-channel beta-modulation `.npz` recording and its saved labels.

Only one task is used in this notebook:

- `Idle vs Beta`

Label convention for the loaded labeled file:

- `0 = Idle`
- `1 = Beta`


In [22]:
from pathlib import Path
import subprocess
import sys

# =============================================================================
# Data loading: one continuous channel + sample-level labels only
# =============================================================================
# Filepaths are intentionally preserved.
SCRIPT_PATH = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/label.py')
INPUT_NPZ = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/cont_data.npz')
OUTPUT_NPZ = Path('/content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/cont_data_labeled.npz')
# SCRIPT_PATH = Path('/content/drive/MyDrive/BCI/online classification/signal_generator/label.py')
# INPUT_NPZ = Path('/content/drive/MyDrive/BCI/online classification/signal_generator/cont_data.npz')
# OUTPUT_NPZ = Path('/content/drive/MyDrive/BCI/online classification/signal_generator/cont_data_labeled2.npz')

# Loaded-label convention used by this notebook:
#   label 0 -> Idle
#   label 1 -> Beta
IDLE_LABEL = 0
BETA_LABEL = 1
N_CHANNELS = 1

_LABELED_DATA_CACHE = None
_SPLIT_CACHE = {}


def _path_for_runtime(path: Path) -> Path:
    """Use the Drive path in Colab; fall back to /mnt/data during local inspection."""
    path = Path(path)
    if path.exists():
        return path
    local_fallback = Path('/mnt/data') / path.name
    if local_fallback.exists():
        return local_fallback
    # Local compatibility for earlier artifact name. This does not change the Drive filepath above.
    if path.name == 'cont_data_labeled2.npz':
        legacy_local = Path('/mnt/data/cont_data_labeled.npz')
        if legacy_local.exists():
            return legacy_local
    return path


def ensure_labeled_beta_file(force=False):
    """
    Return the labeled .npz path.

    The current notebook expects a relabeled file with sample-level labels:
        eeg
        samplerate
        sample_labels

    If the labeled file is missing, this function optionally calls the existing
    label.py script using the preserved filepaths.
    """
    script_path = _path_for_runtime(SCRIPT_PATH)
    input_npz = _path_for_runtime(INPUT_NPZ)
    output_npz = _path_for_runtime(OUTPUT_NPZ)

    needs_labeling = force or (not output_npz.exists())
    if output_npz.exists() and input_npz.exists():
        needs_labeling = needs_labeling or (input_npz.stat().st_mtime > output_npz.stat().st_mtime)

    if needs_labeling:
        if not script_path.exists():
            raise FileNotFoundError(
                f'Labeled file not found and labeling script not found.\n'
                f'Labeled file: {output_npz}\n'
                f'Labeling script: {script_path}'
            )
        if not input_npz.exists():
            raise FileNotFoundError(f'Input .npz not found: {input_npz}')

        cmd = [sys.executable, str(script_path), str(input_npz), '--output-npz', str(output_npz)]
        print('Running labeling command:')
        print(' '.join(cmd))
        completed = subprocess.run(cmd, capture_output=True, text=True, check=True)
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
    else:
        print(f'Using existing labeled file: {output_npz}')

    return output_npz


def load_labeled_beta_data(force_reload=False):
    """Load the saved single-channel Idle-vs-Beta recording with sample-level labels."""
    global _LABELED_DATA_CACHE
    if _LABELED_DATA_CACHE is not None and not force_reload:
        return _LABELED_DATA_CACHE

    output_npz = ensure_labeled_beta_file(force=False)
    labeled = np.load(output_npz)

    required = {'eeg', 'samplerate', 'sample_labels'}
    missing = required.difference(set(labeled.files))
    if missing:
        raise KeyError(f'Labeled file is missing required key(s): {sorted(missing)}. Found keys: {labeled.files}')

    eeg = np.asarray(labeled['eeg'], dtype=np.float32)
    fs = int(np.asarray(labeled['samplerate']).item())
    sample_labels = np.asarray(labeled['sample_labels'], dtype=np.int64).reshape(-1)

    # Normalize EEG to shape (1, samples). This notebook is intentionally one-channel only.
    if eeg.ndim == 1:
        eeg = eeg[None, :]
    elif eeg.ndim == 2:
        if eeg.shape[0] > eeg.shape[1]:
            eeg = eeg.T
        eeg = eeg[:1, :]
    else:
        raise ValueError(f'Expected 1D or 2D EEG array, got shape {eeg.shape}')

    keep_samples = min(eeg.shape[-1], len(sample_labels))
    eeg = eeg[:, :keep_samples]
    sample_labels = sample_labels[:keep_samples]

    if not set(np.unique(sample_labels).tolist()).issubset({IDLE_LABEL, BETA_LABEL}):
        raise ValueError('sample_labels must contain only labels 0 and 1.')

    _LABELED_DATA_CACHE = {
        'eeg': eeg.astype(np.float32),
        'sample_labels': sample_labels.astype(np.int64),
        'fs': fs,
        'source_path': str(output_npz),
        'label_convention': '0=Idle, 1=Beta',
    }
    print(f"Loaded {keep_samples} samples from {output_npz} at fs={fs} Hz")
    print('Label counts:', dict(zip(*np.unique(sample_labels, return_counts=True))))
    return _LABELED_DATA_CACHE


def _as_channel_matrix(eeg, n_channels=N_CHANNELS):
    """Return data as (1, n_samples). No channel repetition is used."""
    arr = np.asarray(eeg, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None, :]
    elif arr.ndim == 2:
        if arr.shape[0] > arr.shape[1]:
            arr = arr.T
        arr = arr[:1, :]
    else:
        raise ValueError(f'Expected 1D or 2D EEG array, got shape {arr.shape}')
    if n_channels != 1:
        raise ValueError('This notebook is configured for one input channel only.')
    return arr.astype(np.float32)


def split_labeled_beta_data(cfg=None, train_fraction=None, n_channels=N_CHANNELS):
    """Split the continuous labeled recording by sample index."""
    if n_channels != 1:
        raise ValueError('This notebook is configured for one input channel only.')

    source = load_labeled_beta_data()
    if train_fraction is None:
        train_fraction = getattr(cfg, 'train_fraction', 0.75) if cfg is not None else 0.75

    cache_key = (float(train_fraction), n_channels, source['source_path'])
    if cache_key in _SPLIT_CACHE:
        return _SPLIT_CACHE[cache_key]

    data_all = _as_channel_matrix(source['eeg'], n_channels=1)
    n_samples = data_all.shape[1]
    split_sample = int(round(n_samples * float(train_fraction)))
    split_sample = max(1, min(n_samples - 1, split_sample))

    split = {
        'train': {
            'data': data_all[:, :split_sample].copy(),
            'sample_labels': source['sample_labels'][:split_sample].copy(),
        },
        'test': {
            'data': data_all[:, split_sample:].copy(),
            'sample_labels': source['sample_labels'][split_sample:].copy(),
        },
        'fs': source['fs'],
        'source_path': source['source_path'],
        'train_fraction': float(train_fraction),
        'split_sample': int(split_sample),
        'n_samples': int(n_samples),
        'label_convention': source['label_convention'],
    }
    _SPLIT_CACHE[cache_key] = split
    return split


def resolve_channel_values(values, n_channels=N_CHANNELS, name='values'):
    """Return one numeric value for the single loaded channel."""
    if np.isscalar(values):
        return np.array([float(values)], dtype=np.float32)
    arr = np.asarray(values, dtype=np.float32).reshape(-1)
    if len(arr) == 0:
        raise ValueError(f'Expected at least one value for {name}.')
    return np.array([float(arr[0])], dtype=np.float32)


def amplitude_key(amplitude):
    if np.isscalar(amplitude):
        return float(amplitude)
    return tuple(float(x) for x in np.asarray(amplitude, dtype=float).reshape(-1))


def format_amplitude(amplitude):
    if np.isscalar(amplitude):
        return f'{float(amplitude):g}'
    amps = np.asarray(amplitude, dtype=float).reshape(-1)
    return '(' + ', '.join(f'{a:g}' for a in amps) + ')'


def _mean_abs_by_label(data, sample_labels, label):
    mask = np.asarray(sample_labels) == int(label)
    if not np.any(mask):
        return tuple(float('nan') for _ in range(data.shape[0]))
    return tuple(float(x) for x in np.mean(np.abs(data[:, mask]), axis=1))


def build_continuous_binary_dataset(split='train', cfg=None):
    """
    Return the loaded continuous one-channel data for a requested split.

    This is a compatibility wrapper for plotting and metadata. It does not create
    synthetic data.
    """
    split_data = split_labeled_beta_data(cfg=cfg, n_channels=1)
    if split not in {'train', 'test'}:
        raise ValueError("split must be 'train' or 'test'.")
    ds = split_data[split]
    data = np.asarray(ds['data'], dtype=np.float32)
    sample_labels_out = np.asarray(ds['sample_labels'], dtype=np.int64)

    idle_abs = _mean_abs_by_label(data, sample_labels_out, IDLE_LABEL)
    beta_abs = _mean_abs_by_label(data, sample_labels_out, BETA_LABEL)

    return {
        'data': data,
        'sample_labels': sample_labels_out,
        'fs': int(split_data['fs']),
        'source_path': split_data['source_path'],
        'label_convention': '0=Idle, 1=Beta',
        'idle_frequency_hz': ('loaded',),
        'active_frequency_hz': ('loaded',),
        'frequency_hz': ('loaded',),
        'idle_channel_amps': idle_abs,
        'active_channel_amps': beta_abs,
        'signal_noise_std': tuple(float(x) for x in resolve_channel_values((0.0,), n_channels=1, name='noise stds')),
    }


In [23]:
def preview_task(
    task_name,
    active_amp=None,
    active_frequency_hz=None,
    idle_frequency_hz=None,
    fs=None,
    duration_sec=8.0,
    idle_amp=None,
    seed=None,
    signal_noise_std=0.0,
    feature_names=None,
    split='train',
):
    """Preview the loaded one-channel Idle-vs-Beta signal and sample-level labels on the same axes."""
    if feature_names is None:
        feature_names = ['channel 1']

    ds = build_continuous_binary_dataset(split=split, cfg=globals().get('cfg', None))
    x = ds['data']
    sample_labels = ds['sample_labels']
    fs_loaded = int(ds['fs'])

    n = min(x.shape[1], int(round(float(duration_sec) * fs_loaded)))
    t = np.arange(n) / fs_loaded

    fig, ax = plt.subplots(1, 1, figsize=(12, 3.8))
    ax.plot(t, x[0, :n], linewidth=0.9, label='channel 1')
    ax.step(t, sample_labels[:n], where='post', linewidth=1.2, alpha=0.85, label='label: 0=Idle, 1=Beta')

    ax.set_title(f'{task_name} preview | loaded one-channel data | {split} split')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('EEG amplitude / label value')
    ax.set_yticks([0, 1])
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Sliding-window extraction and relabeling


In [24]:
def relabel_window(window_labels, mode="endpoint"):
    """Assign a window label from sample-level labels."""
    window_labels = np.asarray(window_labels, dtype=np.int64)
    if mode == "endpoint":
        return int(window_labels[-1])
    if mode == "majority":
        counts = np.bincount(window_labels, minlength=2)
        return int(np.argmax(counts))
    raise ValueError("label_mode must be 'endpoint' or 'majority'.")


def compute_bandpower_fft_1d(x, fs, band=(18.0, 22.0), eps=1e-12):
    """Compute log10 FFT band power for one 1D window."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    x = x - np.mean(x)
    taper = np.hanning(len(x)).astype(np.float32)
    xw = x * taper
    freqs = np.fft.rfftfreq(len(xw), d=1.0 / fs)
    fft_vals = np.fft.rfft(xw)
    power = np.abs(fft_vals) ** 2
    band = tuple(float(v) for v in band)
    mask = (freqs >= band[0]) & (freqs <= band[1])
    if not np.any(mask):
        raise ValueError(f'No FFT bins found inside band={band} for window length={len(x)} and fs={fs}.')
    return float(np.log10(np.sum(power[mask]) + eps))


def make_sliding_windows(data, sample_labels, fs, window_sec, stride_sec, label_mode="endpoint", bandpower_hz=(18.0, 22.0)):
    """
    Create sliding windows directly from the continuous sample stream.

    For each 1-second sliding window, the notebook:
      1. keeps the raw waveform window for visualization,
      2. computes one FFT log-band-power feature for training,
      3. assigns a label from the sample-level labels.

    Returns:
      X_features  : (n_windows, 1, 1), FFT log-band-power features for the LSTM
      y           : (n_windows,)
      end_samples : 1-based end sample for each causal window
      X_raw       : (n_windows, window_samples, 1), raw signal windows for plotting only
    """
    if label_mode not in {"endpoint", "majority"}:
        raise ValueError("label_mode must be 'endpoint' or 'majority'.")

    data = _as_channel_matrix(data, n_channels=1)
    sample_labels = np.asarray(sample_labels, dtype=np.int64).reshape(-1)
    n = min(data.shape[1], len(sample_labels))
    data = data[:, :n]
    sample_labels = sample_labels[:n]

    win = int(round(window_sec * fs))
    stride = int(round(stride_sec * fs))
    if win <= 0 or stride <= 0:
        raise ValueError('window_sec and stride_sec must produce positive sample counts.')
    if n < win:
        raise ValueError(f'Not enough samples ({n}) for window length ({win}).')

    X_features, X_raw, y, end_samples = [], [], [], []
    for end in range(win, n + 1, stride):
        start = end - win
        raw_window = data[:, start:end].T.astype(np.float32)  # (time, 1)
        feature = compute_bandpower_fft_1d(raw_window[:, 0], fs=fs, band=bandpower_hz)
        X_features.append([[feature]])  # one time step, one feature
        X_raw.append(raw_window)
        y.append(relabel_window(sample_labels[start:end], mode=label_mode))
        end_samples.append(end)

    return (
        np.asarray(X_features, dtype=np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(end_samples, dtype=int),
        np.asarray(X_raw, dtype=np.float32),
    )


## Dataset, normalization, and interpretable LSTM


In [25]:
class WindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def fit_normalizer(X_train):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def apply_normalizer(X, mean, std):
    return ((X - mean) / std).astype(np.float32)


class InterpretableLSTMCell(nn.Module):
    """
    Transparent LSTM cell with PyTorch-compatible gate ordering:
        [input_gate, forget_gate, candidate_gate, output_gate]

    The cell returns both the updated states and the raw/post-activation gate
    tensors needed for interpretability and GIF generation.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = int(input_size)
        self.hidden_size = int(hidden_size)
        self.num_gates = 4

        self.weight_ih = nn.Parameter(torch.empty(self.num_gates * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(self.num_gates * hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(self.num_gates * hidden_size))
        self.bias_hh = nn.Parameter(torch.empty(self.num_gates * hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        # Match the default nn.LSTM/LSTMCell initialization style.
        bound = 1.0 / math.sqrt(self.hidden_size)
        for param in self.parameters():
            nn.init.uniform_(param, -bound, bound)

    def forward(self, x_t, h_prev, c_prev):
        gate_pre = (
            torch.matmul(x_t, self.weight_ih.t())
            + torch.matmul(h_prev, self.weight_hh.t())
            + self.bias_ih
            + self.bias_hh
        )
        i_pre, f_pre, g_pre, o_pre = gate_pre.chunk(self.num_gates, dim=1)

        i_t = torch.sigmoid(i_pre)
        f_t = torch.sigmoid(f_pre)
        g_t = torch.tanh(g_pre)       # candidate cell update, usually written c_tilde_t
        o_t = torch.sigmoid(o_pre)

        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)

        gates = {
            "input_gate_pre": i_pre,
            "forget_gate_pre": f_pre,
            "candidate_gate_pre": g_pre,
            "output_gate_pre": o_pre,
            "input_gate": i_t,
            "forget_gate": f_t,
            "candidate_gate": g_t,
            "output_gate": o_t,
            "cell_state": c_t,
            "hidden_state": h_t,
        }
        return h_t, c_t, gates


class InterpretableLSTM(nn.Module):
    """
    Explicitly unrolled multi-layer LSTM.

    When return_gates=True, returns matrices with shape (batch, time, hidden):
        gate_log[layer_idx]["input_gate"]      -> i_t over time
        gate_log[layer_idx]["forget_gate"]     -> f_t over time
        gate_log[layer_idx]["candidate_gate"]  -> c_tilde_t over time
        gate_log[layer_idx]["output_gate"]     -> o_t over time
        gate_log[layer_idx]["cell_state"]      -> c_t over time
        gate_log[layer_idx]["hidden_state"]    -> h_t over time
    """
    def __init__(self, input_size=3, hidden_size=4, num_layers=1, batch_first=True):
        super().__init__()
        if not batch_first:
            raise ValueError("This notebook expects batch_first=True.")
        self.input_size = int(input_size)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.batch_first = bool(batch_first)

        cells = []
        for layer_idx in range(self.num_layers):
            layer_input_size = self.input_size if layer_idx == 0 else self.hidden_size
            cells.append(InterpretableLSTMCell(layer_input_size, self.hidden_size))
        self.cells = nn.ModuleList(cells)

    def get_weight_ih(self, layer_idx=0):
        return self.cells[layer_idx].weight_ih

    def get_weight_hh(self, layer_idx=0):
        return self.cells[layer_idx].weight_hh

    def get_bias_ih(self, layer_idx=0):
        return self.cells[layer_idx].bias_ih

    def get_bias_hh(self, layer_idx=0):
        return self.cells[layer_idx].bias_hh

    def forward(self, x, return_gates=False):
        if x.ndim != 3:
            raise ValueError(f"Expected x with shape (batch, time, channels), got {tuple(x.shape)}")

        batch_size, seq_len, _ = x.shape
        layer_input = x
        h_n, c_n = [], []
        all_gate_logs = {}

        for layer_idx, cell in enumerate(self.cells):
            h_t = x.new_zeros(batch_size, self.hidden_size)
            c_t = x.new_zeros(batch_size, self.hidden_size)

            layer_outputs = []
            layer_gate_lists = {
                "input_gate_pre": [],
                "forget_gate_pre": [],
                "candidate_gate_pre": [],
                "output_gate_pre": [],
                "input_gate": [],
                "forget_gate": [],
                "candidate_gate": [],
                "output_gate": [],
                "cell_state": [],
                "hidden_state": [],
            }

            for t in range(seq_len):
                x_t = layer_input[:, t, :]
                h_t, c_t, gates = cell(x_t, h_t, c_t)
                layer_outputs.append(h_t)

                if return_gates:
                    for name, value in gates.items():
                        layer_gate_lists[name].append(value)

            layer_output = torch.stack(layer_outputs, dim=1)
            layer_input = layer_output
            h_n.append(h_t)
            c_n.append(c_t)

            if return_gates:
                all_gate_logs[layer_idx] = {
                    name: torch.stack(values, dim=1)
                    for name, values in layer_gate_lists.items()
                }

        h_n = torch.stack(h_n, dim=0)
        c_n = torch.stack(c_n, dim=0)

        if return_gates:
            return layer_input, (h_n, c_n), all_gate_logs
        return layer_input, (h_n, c_n)


class LSTMClassifier(nn.Module):
    def __init__(self, input_size=3, hidden_size=4, num_layers=1, num_classes=2):
        super().__init__()
        self.lstm = InterpretableLSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, return_gates=False):
        if return_gates:
            out, _, gate_logs = self.lstm(x, return_gates=True)
            h_last = out[:, -1, :]
            logits = self.fc(h_last)
            return logits, gate_logs

        out, _ = self.lstm(x, return_gates=False)
        h_last = out[:, -1, :]
        return self.fc(h_last)

    @torch.no_grad()
    def extract_gate_matrices(self, X, layer_idx=0, device=None):
        """
        Return NumPy gate/state matrices for one sample or a batch.

        Output arrays have shape:
            (n_windows, n_time_samples, hidden_size)
        """
        was_training = self.training
        self.eval()

        if isinstance(X, torch.Tensor):
            x = X.detach().float()
        else:
            x = torch.tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32)
        if x.ndim == 2:
            x = x.unsqueeze(0)
        if x.ndim != 3:
            raise ValueError(f"Expected X with shape (time, channels) or (batch, time, channels), got {tuple(x.shape)}")

        if device is None:
            device = next(self.parameters()).device
        x = x.to(device)
        _, gate_logs = self(x, return_gates=True)
        layer_logs = gate_logs[int(layer_idx)]

        out = {name: value.detach().cpu().numpy().astype(np.float32) for name, value in layer_logs.items()}

        if was_training:
            self.train()
        return out


In [26]:
GATE_MATRIX_KEYS = [
    "input_gate_pre",
    "forget_gate_pre",
    "candidate_gate_pre",
    "output_gate_pre",
    "input_gate",
    "forget_gate",
    "candidate_gate",
    "output_gate",
    "cell_state",
    "hidden_state",
]


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds, all_probs, all_true = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            total_loss += loss.item() * len(yb)
            total_correct += int((preds == yb).sum().item())
            total += len(yb)
            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_true.append(yb.cpu().numpy())
    return {
        "loss": total_loss / max(total, 1),
        "acc": total_correct / max(total, 1),
        "preds": np.concatenate(all_preds),
        "probs": np.concatenate(all_probs),
        "true": np.concatenate(all_true),
    }


def select_balanced_probe_indices(y, max_windows_per_class=8):
    """
    Deterministically select the first N windows from each class.

    These fixed probe windows are reused after every epoch, so changes in
    gate/state matrices reflect learning rather than changing input samples.
    """
    y_np = np.asarray(y, dtype=np.int64)
    if max_windows_per_class is None:
        return np.arange(len(y_np), dtype=np.int64)

    selected = []
    for cls in np.unique(y_np):
        cls_idx = np.where(y_np == cls)[0]
        selected.append(cls_idx[:int(max_windows_per_class)])
    if not selected:
        return np.array([], dtype=np.int64)
    return np.concatenate(selected).astype(np.int64)


@torch.no_grad()
def extract_epoch_gate_snapshot(model, X_probe, y_probe, epoch, probe_window_idx=None, layer_idx=0):
    """
    Extract one epoch's fixed-probe gate/state matrices.

    Gate/state arrays have shape:
        (probe_windows, time_samples, hidden_size)
    """
    was_training = model.training
    model.eval()

    x = torch.tensor(np.asarray(X_probe, dtype=np.float32), dtype=torch.float32, device=next(model.parameters()).device)
    logits, gate_logs = model(x, return_gates=True)
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy().astype(np.float32)
    preds = logits.argmax(dim=1).detach().cpu().numpy().astype(np.int64)
    layer_logs = gate_logs[int(layer_idx)]

    snapshot = {
        "epoch": int(epoch),
        "probe_window_idx": np.asarray(probe_window_idx if probe_window_idx is not None else np.arange(len(X_probe)), dtype=np.int64),
        "X_probe": np.asarray(X_probe, dtype=np.float32),
        "y_true": np.asarray(y_probe, dtype=np.int64),
        "probs": probs,
        "y_pred": preds,
    }
    for key in GATE_MATRIX_KEYS:
        snapshot[key] = layer_logs[key].detach().cpu().numpy().astype(np.float32)

    if was_training:
        model.train()
    return snapshot


def stack_epoch_gate_snapshots(epoch_snapshots):
    """
    Stack per-epoch snapshots into GIF-ready arrays.

    Final gate/state shape:
        (epochs, probe_windows, time_samples, hidden_size)
    """
    if epoch_snapshots is None or len(epoch_snapshots) == 0:
        return None

    first = epoch_snapshots[0]
    export = {
        "epochs": np.asarray([snap["epoch"] for snap in epoch_snapshots], dtype=np.int64),
        "probe_window_idx": first["probe_window_idx"].astype(np.int64),
        "X_probe": first["X_probe"].astype(np.float32),
        "y_true": first["y_true"].astype(np.int64),
        "probs": np.stack([snap["probs"] for snap in epoch_snapshots], axis=0).astype(np.float32),
        "y_pred": np.stack([snap["y_pred"] for snap in epoch_snapshots], axis=0).astype(np.int64),
    }
    for key in GATE_MATRIX_KEYS:
        export[key] = np.stack([snap[key] for snap in epoch_snapshots], axis=0).astype(np.float32)
    return export


def train_lstm(
    X_train,
    y_train,
    X_test,
    y_test,
    cfg,
    seed,
    epoch_gate_probe=None,
    log_epoch_gates=False,
    epoch_gate_layer_idx=0,
):
    set_global_seed(seed)
    model = LSTMClassifier(input_size=X_train.shape[-1], hidden_size=cfg.hidden_size, num_layers=cfg.num_layers, num_classes=2).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=cfg.batch_size, shuffle=True, generator=generator, num_workers=0)
    test_loader = DataLoader(WindowDataset(X_test, y_test), batch_size=cfg.batch_size, shuffle=False, num_workers=0)

    epoch_gate_snapshots = []
    if log_epoch_gates:
        if epoch_gate_probe is None:
            raise ValueError("log_epoch_gates=True requires epoch_gate_probe.")
        probe_X = np.asarray(epoch_gate_probe["X"], dtype=np.float32)
        probe_y = np.asarray(epoch_gate_probe["y"], dtype=np.int64)
        probe_idx = np.asarray(epoch_gate_probe.get("probe_window_idx", np.arange(len(probe_y))), dtype=np.int64)
    else:
        probe_X = probe_y = probe_idx = None

    history = []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(yb)
            train_correct += int((logits.argmax(dim=1) == yb).sum().item())
            train_total += len(yb)

        test_stats = evaluate_model(model, test_loader, criterion)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss / max(train_total, 1),
            "train_acc": train_correct / max(train_total, 1),
            "test_loss": test_stats["loss"],
            "test_acc": test_stats["acc"],
        })

        if log_epoch_gates:
            epoch_gate_snapshots.append(
                extract_epoch_gate_snapshot(
                    model=model,
                    X_probe=probe_X,
                    y_probe=probe_y,
                    epoch=epoch,
                    probe_window_idx=probe_idx,
                    layer_idx=epoch_gate_layer_idx,
                )
            )

    return model, pd.DataFrame(history), epoch_gate_snapshots


## End-to-end window experiment


In [27]:
def prepare_windowed_binary_dataset(task_name, active_frequency_hz, idle_frequency_hz, active_amp, label_mode, cfg, seed):
    """
    Load continuous one-channel train/test recordings, then create sliding-window
    FFT band-power features directly from sample-level labels.
    """
    split = split_labeled_beta_data(cfg=cfg, n_channels=1)

    train_ds = build_continuous_binary_dataset(split='train', cfg=cfg)
    test_ds = build_continuous_binary_dataset(split='test', cfg=cfg)

    X_train, y_train, train_end_samples, X_train_raw = make_sliding_windows(
        train_ds['data'], train_ds['sample_labels'], split['fs'], cfg.window_sec, cfg.stride_sec,
        label_mode=label_mode, bandpower_hz=cfg.bandpower_hz,
    )
    X_test, y_test, test_end_samples, X_test_raw = make_sliding_windows(
        test_ds['data'], test_ds['sample_labels'], split['fs'], cfg.window_sec, cfg.stride_sec,
        label_mode=label_mode, bandpower_hz=cfg.bandpower_hz,
    )

    train_window_samples = int(round(cfg.window_sec * split['fs']))
    test_window_samples = int(round(cfg.window_sec * split['fs']))
    train_start_samples = train_end_samples - train_window_samples
    test_start_samples = test_end_samples - test_window_samples

    mean, std = fit_normalizer(X_train)
    X_train_norm = apply_normalizer(X_train, mean, std)
    X_test_norm = apply_normalizer(X_test, mean, std)

    return {
        'task': task_name,
        'label_mode': label_mode,
        'active_amp': active_amp,
        'active_amp_label': format_amplitude(active_amp),
        'idle_amp': cfg.idle_amp,
        'idle_amp_label': format_amplitude(cfg.idle_amp),
        'active_channel_amps': train_ds['active_channel_amps'],
        'idle_channel_amps': train_ds['idle_channel_amps'],
        'idle_frequency_hz': train_ds['idle_frequency_hz'],
        'active_frequency_hz': train_ds['active_frequency_hz'],
        'signal_noise_std': train_ds['signal_noise_std'],
        'bandpower_hz': tuple(float(x) for x in cfg.bandpower_hz),
        'train_fs': split['fs'],
        'test_fs': split['fs'],
        'train_ds': train_ds,
        'test_ds': test_ds,
        'X_train_raw': X_train_raw,      # raw waveform windows for visualization only
        'X_test_raw': X_test_raw,        # raw waveform windows for visualization only
        'X_train_bandpower': X_train,
        'X_test_bandpower': X_test,
        'X_train': X_train_norm,         # normalized band-power features used for training
        'y_train': y_train,
        'X_test': X_test_norm,
        'y_test': y_test,
        'train_start_samples': train_start_samples,
        'train_end_samples': train_end_samples,
        'test_start_samples': test_start_samples,
        'test_end_samples': test_end_samples,
        'test_t_sec': test_end_samples / split['fs'],
        'normalizer_mean': mean,
        'normalizer_std': std,
        'train_window_count': len(y_train),
        'test_window_count': len(y_test),
        'source_path': split['source_path'],
        'train_fraction': split['train_fraction'],
        'feature_description': 'single FFT log-band-power feature per sliding window',
    }


def train_and_evaluate_prepared_window_dataset(prepared, cfg, seed):
    """Step 5: train on prepared band-power windows and evaluate on held-out test windows."""
    run_epoch_gate_export = bool(globals().get('RUN_EPOCH_GATE_MATRIX_EXPORT', False))
    probe_split = globals().get('EPOCH_GATE_PROBE_SPLIT', 'test')
    max_probe_per_class = globals().get('MAX_EPOCH_GATE_PROBE_WINDOWS_PER_CLASS', 8)

    epoch_gate_probe = None
    if run_epoch_gate_export:
        if probe_split == 'train':
            probe_source_X = prepared['X_train']
            probe_source_y = prepared['y_train']
        elif probe_split == 'test':
            probe_source_X = prepared['X_test']
            probe_source_y = prepared['y_test']
        else:
            raise ValueError("EPOCH_GATE_PROBE_SPLIT must be 'train' or 'test'.")

        probe_idx = select_balanced_probe_indices(probe_source_y, max_windows_per_class=max_probe_per_class)
        epoch_gate_probe = {
            'split': probe_split,
            'probe_window_idx': probe_idx,
            'X': np.asarray(probe_source_X, dtype=np.float32)[probe_idx],
            'y': np.asarray(probe_source_y, dtype=np.int64)[probe_idx],
        }

    model, history, epoch_gate_snapshots = train_lstm(
        prepared['X_train'], prepared['y_train'],
        prepared['X_test'], prepared['y_test'],
        cfg, seed=seed,
        epoch_gate_probe=epoch_gate_probe,
        log_epoch_gates=run_epoch_gate_export,
        epoch_gate_layer_idx=0,
    )

    criterion = nn.CrossEntropyLoss()
    test_loader = DataLoader(
        WindowDataset(prepared['X_test'], prepared['y_test']),
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_stats = evaluate_model(model, test_loader, criterion)

    run = dict(prepared)
    run.update({
        'model': model,
        'history': history,
        'test_acc': test_stats['acc'],
        'test_true': test_stats['true'],
        'test_preds': test_stats['preds'],
        'test_probs': test_stats['probs'],
        'epoch_gate_probe': epoch_gate_probe,
        'epoch_gate_snapshots': epoch_gate_snapshots,
    })
    return run


def run_one_experiment(task_name, active_frequency_hz, idle_frequency_hz, active_amp, label_mode, cfg, seed):
    """Backward-compatible wrapper: prepare windows, then train/evaluate."""
    prepared = prepare_windowed_binary_dataset(
        task_name=task_name,
        active_frequency_hz=active_frequency_hz,
        idle_frequency_hz=idle_frequency_hz,
        active_amp=active_amp,
        label_mode=label_mode,
        cfg=cfg,
        seed=seed,
    )
    return train_and_evaluate_prepared_window_dataset(prepared, cfg=cfg, seed=seed)


## Experiment runner

The function below is called from the final control cell.


In [28]:
def prepare_window_experiments(cfg, tasks, label_modes, seed):
    """Create band-power sliding-window datasets for every task/mode."""
    if not set(label_modes).issubset({'endpoint', 'majority'}):
        raise ValueError("LABEL_MODES must contain only 'endpoint' and/or 'majority'.")

    prepared_runs = {}
    for task_name, active_frequency_hz, idle_frequency_hz in tasks:
        for label_mode in label_modes:
            prepared = prepare_windowed_binary_dataset(
                task_name=task_name,
                active_frequency_hz=active_frequency_hz,
                idle_frequency_hz=idle_frequency_hz,
                active_amp=cfg.active_amps,
                label_mode=label_mode,
                cfg=cfg,
                seed=seed,
            )
            prepared_runs[(task_name, label_mode)] = prepared
    return prepared_runs


def train_prepared_window_experiments(prepared_runs, cfg, seed):
    """Train/evaluate LSTMs from already prepared band-power window datasets."""
    runs = {}
    rows = []

    for key, prepared in prepared_runs.items():
        run = train_and_evaluate_prepared_window_dataset(prepared, cfg=cfg, seed=seed)
        runs[key] = run
        rows.append({
            'task': run['task'],
            'label_mode': run['label_mode'],
            'bandpower_hz': run.get('bandpower_hz'),
            'feature_description': run.get('feature_description'),
            'train_windows': len(run['y_train']),
            'test_windows': len(run['y_test']),
            'window_shape_train': list(run['X_train'].shape),
            'window_shape_test': list(run['X_test'].shape),
            'test_acc': run['test_acc'],
        })

    results_df = pd.DataFrame(rows)
    return runs, results_df


def run_window_experiments(cfg, tasks, label_modes, seed):
    """Backward-compatible one-call runner."""
    prepared_runs = prepare_window_experiments(cfg, tasks, label_modes, seed)
    return train_prepared_window_experiments(prepared_runs, cfg=cfg, seed=seed)


## Accuracy and prediction plotting functions


In [29]:
def plot_accuracy_and_training_curves(results_df, runs, tasks, label_modes):
    """Plot held-out endpoint-window accuracy and epoch-wise training curves."""
    summary = results_df.pivot_table(index="task", columns="label_mode", values="test_acc")
    display(summary)

    fig, ax = plt.subplots(figsize=(6, 4))
    summary.plot(kind="bar", ax=ax)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Held-out window test accuracy")
    ax.set_title("Window-trained LSTM accuracy using endpoint labels")
    ax.legend(title="Window label")
    plt.xticks(rotation=0)
    plt.show()

    fig, axes = plt.subplots(len(tasks), len(label_modes), figsize=(6 * len(label_modes), 4 * len(tasks)), squeeze=False)
    for row_idx, (task_name, _, _) in enumerate(tasks):
        for col_idx, label_mode in enumerate(label_modes):
            ax = axes[row_idx, col_idx]
            hist = runs[(task_name, label_mode)]["history"]
            ax.plot(hist["epoch"], hist["train_acc"], marker="o", label="train")
            ax.plot(hist["epoch"], hist["test_acc"], marker="o", label="held-out test")
            ax.set_title(f"{task_name} ({label_mode})")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Accuracy")
            ax.set_ylim(0, 1.0)
            ax.legend()
    plt.tight_layout()
    plt.show()



In [30]:
def plot_test_prediction_trace(run, max_windows=300):
    if max_windows is None:
        n = len(run["test_true"])
    else:
        n = min(max_windows, len(run["test_true"]))

    t = run["test_t_sec"][:n]
    y_true = run["test_true"][:n]
    y_pred = run["test_preds"][:n]
    p_active = run["test_probs"][:n, 1]

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.step(t, y_true, where="post", label="true label", linewidth=2)
    ax.step(t, y_pred, where="post", label="predicted label", alpha=0.8)
    ax.plot(t, p_active, label="P(active)", alpha=0.8)
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel("Window end time (s)")
    ax.set_ylabel("Label / probability")
    ax.set_title(f"{run['task']} ({run['label_mode']}) held-out window predictions")
    ax.legend(loc="upper right")
    plt.show()


def plot_all_prediction_traces(runs, max_windows=300):
    """Plot held-out prediction traces for all completed runs."""
    for _, run in runs.items():
        plot_test_prediction_trace(run, max_windows=max_windows)


def plot_window_examples_from_prepared_run(prepared_run, split="train", n_windows=6, feature_names=None):
    """
    Visualize continuous-signal sliding windows.

    The top panel shows the continuous recording with sample-level labels overlaid.
    Shaded spans mark the actual 1-second sliding windows sampled every 0.25 seconds.
    Lower panels show the raw waveform windows used to compute FFT band-power features.
    """
    if feature_names is None:
        feature_names = [f"channel {i + 1}" for i in range(prepared_run[f"X_{split}_raw"].shape[-1])]

    X_raw = prepared_run[f"X_{split}_raw"]
    y = prepared_run[f"y_{split}"]
    ds = prepared_run[f"{split}_ds"]
    fs = prepared_run[f"{split}_fs"] if f"{split}_fs" in prepared_run else prepared_run["train_fs"]
    start_samples = prepared_run[f"{split}_start_samples"]
    end_samples = prepared_run[f"{split}_end_samples"]

    n = min(n_windows, len(y))
    win_t = np.arange(X_raw.shape[1]) / fs

    context_samples = int(round(0.5 * fs))
    continuous_end = min(ds["data"].shape[1], int(end_samples[n - 1]) + context_samples)
    t_cont = np.arange(continuous_end) / fs

    total_rows = 1 + n
    fig, axes = plt.subplots(total_rows, 1, figsize=(12, 2.4 + 2.0 * n), sharex=False, squeeze=False)
    axes = axes.ravel()

    ax = axes[0]
    for ch_idx, feature_name in enumerate(feature_names):
        ax.plot(t_cont, ds["data"][ch_idx, :continuous_end], label=feature_name)
    ax.step(t_cont, ds["sample_labels"][:continuous_end], where='post', linewidth=1.0, alpha=0.8, label='label: 0=Idle, 1=Beta')

    for idx in range(n):
        start_t = start_samples[idx] / fs
        end_t = end_samples[idx] / fs
        ax.axvspan(start_t, end_t, alpha=0.18)
        ax.text(start_t, ax.get_ylim()[1] * 0.88, f"w{idx}", fontsize=8, va="top")

    stride_sec = (end_samples[1] - end_samples[0]) / fs if len(end_samples) > 1 else np.nan
    ax.set_title(
        f"{prepared_run['task']} ({prepared_run['label_mode']}) | continuous {split} signal "
        f"with {prepared_run.get('feature_description', 'window features')} | stride={stride_sec:.2f}s"
    )
    ax.set_xlabel("Time in continuous recording split (s)")
    ax.set_ylabel("Amplitude / label")
    ax.legend(loc="upper right")

    for idx in range(n):
        ax = axes[idx + 1]
        for ch_idx, feature_name in enumerate(feature_names):
            ax.plot(win_t, X_raw[idx, :, ch_idx], label=feature_name if idx == 0 else None)
        bp = prepared_run[f"X_{split}_bandpower"][idx, 0, 0]
        ax.set_title(
            f"Raw window {idx}: {start_samples[idx] / fs:.2f}s–{end_samples[idx] / fs:.2f}s | "
            f"label={'Beta' if y[idx] == 1 else 'Idle'} | log band power={bp:.3f}"
        )
        ax.set_ylabel("Amplitude")
        if idx == 0:
            ax.legend(loc="upper right")

    axes[-1].set_xlabel("Time within extracted 1-second window (s)")
    plt.tight_layout()
    plt.show()


def plot_all_window_examples(prepared_runs, n_windows=6, split="train", feature_names=None):
    """Plot continuous sliding-window examples for all task/mode runs."""
    for _, prepared_run in prepared_runs.items():
        plot_window_examples_from_prepared_run(
            prepared_run,
            split=split,
            n_windows=n_windows,
            feature_names=feature_names,
        )


## LSTM gate/state diagnostics and GIF-ready matrix export


In [31]:
def extract_lstm_layer_diagnostics(model, X, layer_idx=0):
    """
    Extract LSTM gate activations and hidden/cell states directly from the
    custom interpretable LSTM.

    Returned arrays have shape:
        (n_windows, n_time_samples, hidden_size)
    """
    return model.extract_gate_matrices(X, layer_idx=layer_idx, device=DEVICE)


def _get_interpretable_lstm(model):
    if not hasattr(model, "lstm") or not isinstance(model.lstm, InterpretableLSTM):
        raise TypeError("Expected model.lstm to be an InterpretableLSTM instance.")
    return model.lstm


def _get_lstm_weight_ih(model, layer_idx=0):
    lstm = _get_interpretable_lstm(model)
    return lstm.get_weight_ih(layer_idx).detach().cpu().numpy()


def _gate_slices(hidden_size):
    return {
        "input_gate": slice(0, hidden_size),
        "forget_gate": slice(hidden_size, 2 * hidden_size),
        "candidate_gate": slice(2 * hidden_size, 3 * hidden_size),
        "output_gate": slice(3 * hidden_size, 4 * hidden_size),
    }


def summarize_input_weight_by_channel(model, layer_idx=0, feature_names=None):
    """Return mean absolute input-to-gate weight per gate and input channel."""
    lstm = _get_interpretable_lstm(model)
    W_ih = _get_lstm_weight_ih(model, layer_idx=layer_idx)
    hidden_size = lstm.hidden_size
    if feature_names is None:
        feature_names = [f"channel {i + 1}" for i in range(W_ih.shape[1])]

    rows = []
    for gate_name, sl in _gate_slices(hidden_size).items():
        W_gate = W_ih[sl, :]
        for ch_idx, name in enumerate(feature_names):
            rows.append({
                "gate": gate_name,
                "input_channel": name,
                "mean_abs_weight": float(np.mean(np.abs(W_gate[:, ch_idx]))),
            })
    return pd.DataFrame(rows)


def summarize_gate_input_contributions_by_class(model, X, y, layer_idx=0, max_windows_per_class=64):
    """
    Mean abs input contribution |x_ch(t) * W_ih(gate, hidden, ch)| by class.

    Output:
        summary[class][gate][channel] -> array shaped (seq_len, hidden_size)
    """
    lstm = _get_interpretable_lstm(model)
    W_ih = _get_lstm_weight_ih(model, layer_idx=layer_idx)
    hidden_size = lstm.hidden_size
    X_np = np.asarray(X, dtype=np.float32)
    y_np = np.asarray(y, dtype=np.int64)
    feature_names = [f"channel {i + 1}" for i in range(X_np.shape[-1])]

    out = {}
    for cls in np.unique(y_np):
        idx = np.where(y_np == cls)[0]
        if max_windows_per_class is not None:
            idx = idx[:max_windows_per_class]
        X_cls = X_np[idx]
        out[int(cls)] = {}
        for gate_name, sl in _gate_slices(hidden_size).items():
            W_gate = W_ih[sl, :]  # hidden x channels
            out[int(cls)][gate_name] = {}
            for ch_idx, name in enumerate(feature_names):
                contrib = np.abs(X_cls[:, :, ch_idx, None] * W_gate[None, None, :, ch_idx])
                out[int(cls)][gate_name][name] = contrib.mean(axis=0).astype(np.float32)
    return out


def summarize_gate_input_contribution_table(
    contribution_summary,
    feature_names=None,
    class_name_map=None,
):
    """Flatten contribution tensors into one scalar per class × gate × channel."""
    if class_name_map is None:
        class_name_map = {}

    if feature_names is None:
        feature_names = list(next(iter(next(iter(contribution_summary.values())).values())).keys())

    rows = []
    for cls, gate_dict in contribution_summary.items():
        class_name = class_name_map.get(int(cls), f"class_{int(cls)}")
        for gate_name, channel_dict in gate_dict.items():
            gate_values = []
            for channel_name in feature_names:
                mat = np.asarray(channel_dict[channel_name], dtype=np.float32)
                gate_values.append(float(np.mean(mat)))

            gate_total = float(np.sum(gate_values))
            for channel_name, value in zip(feature_names, gate_values):
                rows.append({
                    "class": int(cls),
                    "class_name": class_name,
                    "gate": gate_name,
                    "input_channel": channel_name,
                    "mean_abs_contribution": value,
                    "within_gate_fraction": value / gate_total if gate_total > 0 else 0.0,
                })
    return pd.DataFrame(rows)


def summarize_gate_diagnostics_by_class(model, X, y, layer_idx=0, max_windows_per_class=64):
    X_np = np.asarray(X, dtype=np.float32)
    y_np = np.asarray(y, dtype=np.int64)
    out = {}
    for cls in np.unique(y_np):
        idx = np.where(y_np == cls)[0]
        if max_windows_per_class is not None:
            idx = idx[:max_windows_per_class]
        diag = extract_lstm_layer_diagnostics(model, X_np[idx], layer_idx=layer_idx)
        out[int(cls)] = {k: v.mean(axis=0).astype(np.float32) for k, v in diag.items()}
    return out


def summarize_gate_activation_table(gate_summaries_by_class, class_name_map=None):
    """Return scalar summaries of each gate/state trajectory per class."""
    if class_name_map is None:
        class_name_map = {}

    rows = []
    for cls, summary_dict in gate_summaries_by_class.items():
        class_name = class_name_map.get(int(cls), f"class_{int(cls)}")
        for name, mat in summary_dict.items():
            arr = np.asarray(mat, dtype=np.float32)
            rows.append({
                "class": int(cls),
                "class_name": class_name,
                "quantity": name,
                "mean": float(np.mean(arr)),
                "mean_abs": float(np.mean(np.abs(arr))),
                "std": float(np.std(arr)),
                "min": float(np.min(arr)),
                "max": float(np.max(arr)),
            })
    return pd.DataFrame(rows)


def _time_collapse_to_1_by_hidden(mat, collapse="mean"):
    arr = np.asarray(mat, dtype=np.float32)
    if collapse == "mean":
        return arr.mean(axis=0, keepdims=True).astype(np.float32)
    if collapse == "median":
        return np.median(arr, axis=0, keepdims=True).astype(np.float32)
    if collapse == "max":
        return arr.max(axis=0, keepdims=True).astype(np.float32)
    raise ValueError("collapse must be one of {'mean', 'median', 'max'}")


def summarize_time_collapsed_gate_input_contributions(model, X, y, layer_idx=0, max_windows_per_class=64, collapse="mean"):
    time_resolved = summarize_gate_input_contributions_by_class(model, X, y, layer_idx, max_windows_per_class)
    collapsed = {}
    for cls, gate_dict in time_resolved.items():
        collapsed[int(cls)] = {}
        for gate_name, feature_dict in gate_dict.items():
            collapsed[int(cls)][gate_name] = {
                feature_name: _time_collapse_to_1_by_hidden(mat, collapse=collapse)
                for feature_name, mat in feature_dict.items()
            }
    return collapsed


def combine_time_collapsed_channels(collapsed_summary_dict, feature_names=None):
    """Combine per-channel 1×hidden summaries into one hidden×channel matrix per gate."""
    if feature_names is None:
        feature_names = list(next(iter(collapsed_summary_dict.values())).keys())

    combined = {}
    for gate_name, feature_dict in collapsed_summary_dict.items():
        cols = []
        for feature_name in feature_names:
            arr = np.asarray(feature_dict[feature_name], dtype=np.float32)
            if arr.ndim != 2 or arr.shape[0] != 1:
                raise ValueError(
                    f"Expected collapsed summary for {gate_name}/{feature_name} to have shape (1, hidden_size), got {arr.shape}"
                )
            cols.append(arr[0])
        combined[gate_name] = np.stack(cols, axis=1).astype(np.float32)
    return combined


def build_gate_matrix_export(
    model,
    X,
    y=None,
    probs=None,
    preds=None,
    t_sec=None,
    layer_idx=0,
    max_windows=None,
):
    """
    Build a GIF-ready bundle of gate/state matrices.

    Each gate/state array has shape:
        (selected_windows, time_samples, hidden_size)

    These are the main arrays for an animation:
        input_gate      -> I_t
        forget_gate     -> F_t
        candidate_gate  -> c_tilde_t
        output_gate     -> O_t
        cell_state      -> C_t
        hidden_state    -> H_t
    """
    X_np = np.asarray(X, dtype=np.float32)
    n = X_np.shape[0]
    selected_idx = np.arange(n)
    if max_windows is not None:
        selected_idx = selected_idx[:int(max_windows)]

    gate_mats = extract_lstm_layer_diagnostics(model, X_np[selected_idx], layer_idx=layer_idx)
    export = {
        "selected_window_idx": selected_idx.astype(np.int64),
        "X": X_np[selected_idx].astype(np.float32),
        **gate_mats,
    }

    if y is not None:
        export["y_true"] = np.asarray(y)[selected_idx].astype(np.int64)
    if probs is not None:
        export["probs"] = np.asarray(probs)[selected_idx].astype(np.float32)
    if preds is not None:
        export["y_pred"] = np.asarray(preds)[selected_idx].astype(np.int64)
    if t_sec is not None:
        export["t_sec"] = np.asarray(t_sec)[selected_idx].astype(np.float32)

    return export


def summarize_gate_matrices_by_class(gate_export):
    """Create class-mean gate/state matrices shaped (time_samples, hidden_size)."""
    if "y_true" not in gate_export:
        return {}

    y = np.asarray(gate_export["y_true"])
    keys = [
        "input_gate",
        "forget_gate",
        "candidate_gate",
        "output_gate",
        "cell_state",
        "hidden_state",
    ]
    class_means = {}
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        class_means[int(cls)] = {
            key: np.asarray(gate_export[key], dtype=np.float32)[idx].mean(axis=0).astype(np.float32)
            for key in keys
        }
    return class_means


In [32]:
def run_lstm_diagnostics(
    runs,
    tasks,
    label_modes,
    feature_names,
    class_name_map,
    max_windows_per_class=64,
    show_input_weight_summary=True,
    show_channel_contribution_summary=True,
    show_gate_state_summary=True,
):
    """Run LSTM gate/channel diagnostics as tables only."""
    if list(label_modes) != ["endpoint"]:
        raise ValueError("This notebook supports endpoint relabeling only; use LABEL_MODES = ['endpoint'].")

    all_weight_rows = []
    all_contribution_rows = []
    all_gate_state_rows = []

    for task_name, _, _ in tasks:
        for label_mode in label_modes:
            run = runs[(task_name, label_mode)]
            model = run["model"]
            X_test = run["X_test"]
            y_test = run["y_test"]
            title_prefix = f"{task_name} ({label_mode})"

            print()
            print(f"=== {title_prefix} | three-channel LSTM diagnostics ===")

            if show_input_weight_summary:
                print("Input-to-gate weight summary:")
                weight_summary_df = summarize_input_weight_by_channel(model, layer_idx=0, feature_names=feature_names)
                weight_summary_df["task"] = task_name
                weight_summary_df["label_mode"] = label_mode
                display(weight_summary_df.pivot(index="gate", columns="input_channel", values="mean_abs_weight"))
                all_weight_rows.append(weight_summary_df)

            if show_channel_contribution_summary:
                print("Channel-resolved gate contribution summary:")
                contribution_summary = summarize_gate_input_contributions_by_class(
                    model,
                    X_test,
                    y_test,
                    layer_idx=0,
                    max_windows_per_class=max_windows_per_class,
                )
                contribution_df = summarize_gate_input_contribution_table(
                    contribution_summary,
                    feature_names=feature_names,
                    class_name_map=class_name_map,
                )
                contribution_df["task"] = task_name
                contribution_df["label_mode"] = label_mode
                display(contribution_df)
                all_contribution_rows.append(contribution_df)

            if show_gate_state_summary:
                print("Gate/state activation summary:")
                gate_summaries = summarize_gate_diagnostics_by_class(
                    model,
                    X_test,
                    y_test,
                    layer_idx=0,
                    max_windows_per_class=max_windows_per_class,
                )
                gate_state_df = summarize_gate_activation_table(
                    gate_summaries,
                    class_name_map=class_name_map,
                )
                gate_state_df["task"] = task_name
                gate_state_df["label_mode"] = label_mode
                display(gate_state_df)
                all_gate_state_rows.append(gate_state_df)

    return {
        "weight_summary": pd.concat(all_weight_rows, ignore_index=True) if all_weight_rows else pd.DataFrame(),
        "channel_contribution_summary": pd.concat(all_contribution_rows, ignore_index=True) if all_contribution_rows else pd.DataFrame(),
        "gate_state_summary": pd.concat(all_gate_state_rows, ignore_index=True) if all_gate_state_rows else pd.DataFrame(),
    }


In [33]:
# =============================================================================
# COMPACT INTERNAL LSTM CHANNEL-USE SUMMARY
# =============================================================================

def summarize_channel_gate_internal_use(
    model,
    X,
    y,
    feature_names=None,
    class_name_map=None,
    layer_idx=0,
    max_windows_per_class=64,
):
    """
    Summarize internal LSTM channel use as one scalar per class × channel × gate.

    This uses the channel-resolved gate contribution:

        |x_channel(t) * W_ih_gate_hidden_channel|

    The result is averaged over windows, time samples, and hidden units.
    """
    contribution_summary = summarize_gate_input_contributions_by_class(
        model=model,
        X=X,
        y=y,
        layer_idx=layer_idx,
        max_windows_per_class=max_windows_per_class,
    )
    return summarize_gate_input_contribution_table(
        contribution_summary,
        feature_names=feature_names,
        class_name_map=class_name_map,
    )


def plot_hidden_cell_state_trajectories(
    model,
    X,
    y,
    class_name_map=None,
    layer_idx=0,
    max_windows_per_class=64,
    title_prefix="",
):
    """
    Plot mean |hidden state| and mean |cell state| over time by class.

    These trajectories show whether the LSTM is accumulating, suppressing, or
    exposing class-specific temporal information across the window.
    """
    if class_name_map is None:
        class_name_map = {}

    X_np = np.asarray(X, dtype=np.float32)
    y_np = np.asarray(y, dtype=np.int64)

    fig_h, ax_h = plt.subplots(figsize=(7, 4))
    fig_c, ax_c = plt.subplots(figsize=(7, 4))

    for cls in np.unique(y_np):
        idx = np.where(y_np == cls)[0]
        if max_windows_per_class is not None:
            idx = idx[:max_windows_per_class]
        X_cls = X_np[idx]

        diag = extract_lstm_layer_diagnostics(
            model,
            X_cls,
            layer_idx=layer_idx,
        )

        class_name = class_name_map.get(int(cls), f"class {int(cls)}")

        h_mag = np.mean(np.abs(diag["hidden_state"]), axis=(0, 2))
        c_mag = np.mean(np.abs(diag["cell_state"]), axis=(0, 2))

        ax_h.plot(h_mag, label=class_name)
        ax_c.plot(c_mag, label=class_name)

    ax_h.set_title(f"{title_prefix} | mean absolute hidden-state trajectory")
    ax_h.set_xlabel("Time sample")
    ax_h.set_ylabel("Mean |h_t|")
    ax_h.legend()

    ax_c.set_title(f"{title_prefix} | mean absolute cell-state trajectory")
    ax_c.set_xlabel("Time sample")
    ax_c.set_ylabel("Mean |c_t|")
    ax_c.legend()

    plt.show()


def run_internal_channel_use_summary(
    runs,
    tasks,
    label_modes,
    feature_names,
    class_name_map,
    max_windows_per_class=64,
    plot_state_trajectories=True,
):
    """
    Run compact internal channel-use diagnostics for each trained task.

    Produces:
        1. channel × gate summary dataframe
        2. hidden-state trajectory
        3. cell-state trajectory
    """
    all_rows = []

    for task_name, _, _ in tasks:
        for label_mode in label_modes:
            run = runs[(task_name, label_mode)]

            model = run["model"]
            X_test = run["X_test"]
            y_test = run["y_test"]

            title_prefix = f"{task_name} ({label_mode})"

            print()
            print(f"=== {title_prefix} | compact internal channel-use summary ===")

            summary_df = summarize_channel_gate_internal_use(
                model=model,
                X=X_test,
                y=y_test,
                feature_names=feature_names,
                class_name_map=class_name_map,
                layer_idx=0,
                max_windows_per_class=max_windows_per_class,
            )

            summary_df["task"] = task_name
            summary_df["label_mode"] = label_mode

            display(summary_df)

            if plot_state_trajectories:
                plot_hidden_cell_state_trajectories(
                    model=model,
                    X=X_test,
                    y=y_test,
                    class_name_map=class_name_map,
                    layer_idx=0,
                    max_windows_per_class=max_windows_per_class,
                    title_prefix=title_prefix,
                )

            all_rows.append(summary_df)

    if len(all_rows) > 0:
        return pd.concat(all_rows, ignore_index=True)

    return pd.DataFrame()


# Run Experiments

This notebook now uses one loaded continuous signal and one task only: **Idle vs Beta**.

Key assumptions:

- The labeled `.npz` contains `eeg`, `samplerate`, and sample-level `sample_labels`.
- Label mapping is fixed as `0 = Idle` and `1 = Beta`.
- The recording is split into train/test chunks by sample index.
- Sliding windows are created directly from the continuous signal.
- Each window is converted to an FFT log-band-power feature before training.
- Sliding-window labels are assigned from the sample-level labels using the configured label mode.
- Training and testing are performed on held-out windows, not k-folds.


In [34]:

# =============================================================================
# FINAL CONTROL CELL — RUN ALL REGIMENS AND SAVE EACH TO ITS OWN SUBFOLDER
# =============================================================================

from pathlib import Path
import json

# Reproducibility / device
SEED = 888
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_global_seed(SEED)

# Root directory for all experiment outputs
RESULTS_ROOT = Path("/content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Condition grid
# -----------------------------------------------------------------------------
# The notebook now runs one loaded-data condition only.
CONDITION_REGIMENS = [
    {
        "condition_name": "loaded_cont_data",
        "signal_noise_std": (0.0,),
        "idle_amp": (1.0,),
        "active_amps": (1.0,),
        "purpose": "single-channel labeled recording loaded from the relabeled .npz",
    },
]

# -----------------------------------------------------------------------------
# Rhythm / temporal-pattern regime grid
# -----------------------------------------------------------------------------
# Placeholder metadata only; the actual waveform is loaded from the labeled .npz file.
RHYTHM_REGIMENS = [
    {
        "rhythm_regime_name": "loaded_idle_vs_beta",
        "idle_beta_freq_hz": (20.0,),
        "beta_freq_hz": (20.0,),
        "purpose": "one loaded-data task: Idle vs Beta",
    },
]

# -----------------------------------------------------------------------------
# Core experiment settings shared by all regimens
# -----------------------------------------------------------------------------
BASE_CFG_KWARGS = dict(
    train_fs=200,
    test_fs=200,
    train_fraction=0.75,
    window_sec=1.0,
    stride_sec=0.25,
    bandpower_hz=(18.0, 22.0),
    batch_size=64,
    epochs=12,
    lr=1e-3,
    hidden_size=4,
    num_layers=1,
)

FEATURE_NAMES_1CH = ["channel 1"]
CLASS_NAME_MAP = {0: "Idle", 1: "Beta"}
LABEL_MODES = ["endpoint"]

# Plot / diagnostic switches
RUN_CONTINUOUS_PREVIEWS = True
PREVIEW_DURATION_SEC = 8.0

RUN_WINDOW_EXAMPLE_PLOTS = True
WINDOW_EXAMPLE_SPLIT = "train"
WINDOW_EXAMPLE_N_WINDOWS = 1

PLOT_ACCURACY_AND_TRAINING = True
PLOT_PREDICTION_TRACES = True
PREDICTION_TRACE_MAX_WINDOWS = None

RUN_LSTM_DIAGNOSTICS = True
MAX_WINDOWS_PER_CLASS_FOR_DIAGNOSTICS = None
SHOW_INPUT_WEIGHT_SUMMARY = True
SHOW_CHANNEL_CONTRIBUTION_SUMMARY = True
SHOW_GATE_STATE_SUMMARY = True

# Compact internal channel-use summary: class x channel x gate + hidden/cell trajectories
RUN_INTERNAL_CHANNEL_USE_SUMMARY = True
MAX_WINDOWS_PER_CLASS_FOR_INTERNAL_SUMMARY = None
PLOT_INTERNAL_STATE_TRAJECTORIES = True

# Save explicit I_t/F_t/c_tilde_t/O_t/C_t/H_t matrices from the final trained model.
RUN_GATE_MATRIX_EXPORT = True
MAX_WINDOWS_FOR_GATE_MATRIX_EXPORT = None  # None saves all test windows; use an int for smaller GIF prototypes.

# Save epoch-wise gate/state matrices on a fixed balanced probe set for GIFs over training.
RUN_EPOCH_GATE_MATRIX_EXPORT = True
EPOCH_GATE_PROBE_SPLIT = "test"  # "test" or "train"
MAX_EPOCH_GATE_PROBE_WINDOWS_PER_CLASS = 8  # total probe windows = 2 classes × this value

# -----------------------------------------------------------------------------
# Helpers for looping + saving
# -----------------------------------------------------------------------------
def _slugify(text: str) -> str:
    text = text.replace("|", "_").replace("/", "_")
    text = re.sub(r"\s+", "_", text.strip())
    text = re.sub(r"[^A-Za-z0-9._-]", "", text)
    return text.strip("._") or "run"


def _clean_output_text(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace(" vs ", "_vs_")
    text = text.replace("/", "_")
    text = text.replace("|", "_")
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^a-z0-9._-]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("._") or "item"


def _simple_task_name(task: str) -> str:
    """Create a simpler task stem aligned with the updated gate-visualization notebook."""
    parts = [p.strip() for p in str(task).split("|")]
    if len(parts) >= 3:
        condition, _rhythm, task_name = parts[:3]
        return f"{_clean_output_text(condition)}_{_clean_output_text(task_name)}"
    return _clean_output_text(task)


def _simple_task_stem(task: str, label_mode: str) -> str:
    """Create a simpler output stem aligned with the updated gate-visualization notebook."""
    return f"{_simple_task_name(task)}_{_clean_output_text(label_mode)}"

def _to_builtin(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return [_to_builtin(x) for x in obj]
    if isinstance(obj, list):
        return [_to_builtin(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    return obj

def _save_json(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_builtin(data), f, indent=2)

def _capture_and_save_figures(func, save_dir: Path, stem: str, *args, **kwargs):
    """Capture and save all figures produced by a plotting function in notebook/Colab contexts."""
    save_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = []
    saved_fig_ids = set()
    original_show = plt.show

    def _save_current_figures():
        for fig_num in list(plt.get_fignums()):
            fig = plt.figure(fig_num)
            fig_id = id(fig)
            if fig_id in saved_fig_ids:
                continue
            out_path = save_dir / f"{stem}_{len(saved_paths)+1:02d}.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            saved_paths.append(str(out_path))
            saved_fig_ids.add(fig_id)
            plt.close(fig)

    def _show_and_save(*show_args, **show_kwargs):
        _save_current_figures()
        return original_show(*show_args, **show_kwargs)

    plt.show = _show_and_save
    try:
        out = func(*args, **kwargs)
        _save_current_figures()
        if len(saved_paths) == 0:
            print(f"[WARN] No figures captured for '{stem}' in {save_dir}")
        else:
            print(f"Saved {len(saved_paths)} figure(s) to {save_dir}")
        return out
    finally:
        plt.show = original_show
        plt.close("all")

def build_cfg(condition, rhythm):
    return Config(
        **BASE_CFG_KWARGS,
        idle_beta_freq_hz=rhythm["idle_beta_freq_hz"],
        beta_freq_hz=rhythm["beta_freq_hz"],
        signal_noise_std=condition["signal_noise_std"],
        idle_amp=condition["idle_amp"],
        active_amps=condition["active_amps"],
    )

def build_tasks(cfg, condition_name, rhythm_regime_name):
    return [
        (f"{condition_name} | {rhythm_regime_name} | Idle vs Beta", cfg.beta_freq_hz, cfg.idle_beta_freq_hz),
    ]

def save_run_artifacts(runs, results_df, run_dir: Path):
    run_dir.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(run_dir / "results_summary.csv", index=False)

    pred_dir = run_dir / "prediction_tables"
    hist_dir = run_dir / "training_histories"
    model_dir = run_dir / "model_state_dicts"
    task_output_dir = run_dir / "task_outputs"
    pred_dir.mkdir(parents=True, exist_ok=True)
    hist_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    task_output_dir.mkdir(parents=True, exist_ok=True)

    for key, run in runs.items():
        stem = _simple_task_stem(run["task"], run["label_mode"])
        per_task_dir = task_output_dir / stem
        per_task_dir.mkdir(parents=True, exist_ok=True)

        hist_df = pd.DataFrame(run["history"])
        hist_df.to_csv(hist_dir / f"{stem}.csv", index=False)
        hist_df.to_csv(per_task_dir / "training_history.csv", index=False)

        pred_df = pd.DataFrame({
            "t_sec": run["test_t_sec"],
            "y_true": run["test_true"],
            "y_pred": run["test_preds"],
            "p_idle": run["test_probs"][:, 0],
            "p_beta": run["test_probs"][:, 1],
        })
        pred_df.to_csv(pred_dir / f"{stem}.csv", index=False)
        pred_df.to_csv(per_task_dir / "prediction_table.csv", index=False)

        torch.save(run["model"].state_dict(), model_dir / f"{stem}.pt")
        torch.save(run["model"].state_dict(), per_task_dir / "model_state_dict.pt")

def save_gate_matrix_artifacts(runs, run_dir: Path, max_windows=None):
    """Save explicit gate/state matrices as compressed .npz files for GIF generation."""
    gate_dir = run_dir / "gate_matrix_exports"
    mean_dir = gate_dir / "class_mean_matrices"
    gate_dir.mkdir(parents=True, exist_ok=True)
    mean_dir.mkdir(parents=True, exist_ok=True)

    for key, run in runs.items():
        stem = _simple_task_stem(run["task"], run["label_mode"])

        export = build_gate_matrix_export(
            model=run["model"],
            X=run["X_test"],
            y=run["y_test"],
            probs=run["test_probs"],
            preds=run["test_preds"],
            t_sec=run["test_t_sec"],
            layer_idx=0,
            max_windows=max_windows,
        )
        np.savez_compressed(gate_dir / f"{stem}__gate_matrices.npz", **export)
        per_task_dir = gate_dir / stem
        per_task_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(per_task_dir / "gate_matrices.npz", **export)

        class_means = summarize_gate_matrices_by_class(export)
        flat_means = {}
        for cls, matrices in class_means.items():
            class_label = CLASS_NAME_MAP.get(int(cls), f"class_{int(cls)}")
            class_label = _slugify(class_label)
            for name, mat in matrices.items():
                flat_means[f"{class_label}__{name}"] = mat
        if flat_means:
            np.savez_compressed(mean_dir / f"{stem}__class_mean_gate_matrices.npz", **flat_means)
            np.savez_compressed(per_task_dir / "class_mean_gate_matrices.npz", **flat_means)

        manifest = {
            "task": run["task"],
            "label_mode": run["label_mode"],
            "saved_windows": int(export["X"].shape[0]),
            "X_shape": list(export["X"].shape),
            "gate_shape": list(export["input_gate"].shape),
            "gate_keys": [
                "input_gate_pre",
                "forget_gate_pre",
                "candidate_gate_pre",
                "output_gate_pre",
                "input_gate",
                "forget_gate",
                "candidate_gate",
                "output_gate",
                "cell_state",
                "hidden_state",
            ],
            "notes": {
                "input_gate": "I_t, sigmoid output, range 0 to 1",
                "forget_gate": "F_t, sigmoid output, range 0 to 1",
                "candidate_gate": "c_tilde_t, tanh candidate update, range -1 to 1",
                "output_gate": "O_t, sigmoid output, range 0 to 1",
                "cell_state": "C_t, accumulated memory state",
                "hidden_state": "H_t, exposed/output hidden state",
            },
        }
        _save_json(manifest, gate_dir / f"{stem}__gate_matrices_manifest.json")
        _save_json(manifest, per_task_dir / "gate_matrices_manifest.json")

    print(f"Saved gate matrix exports to {gate_dir}")


def save_epoch_gate_matrix_artifacts(runs, run_dir: Path):
    """Save epoch-wise fixed-probe gate/state matrices for GIFs over training."""
    epoch_dir = run_dir / "epoch_gate_matrix_exports"
    mean_dir = epoch_dir / "class_mean_matrices"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    mean_dir.mkdir(parents=True, exist_ok=True)

    saved_any = False
    for key, run in runs.items():
        snapshots = run.get("epoch_gate_snapshots", None)
        if snapshots is None or len(snapshots) == 0:
            continue

        stem = _simple_task_stem(run["task"], run["label_mode"])

        export = stack_epoch_gate_snapshots(snapshots)
        if export is None:
            continue

        np.savez_compressed(epoch_dir / f"{stem}__epoch_gate_matrices.npz", **export)
        per_task_dir = epoch_dir / stem
        per_task_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(per_task_dir / "epoch_gate_matrices.npz", **export)

        # Class-mean arrays are shaped (epochs, time_samples, hidden_size).
        y_probe = np.asarray(export["y_true"], dtype=np.int64)
        flat_means = {"epochs": export["epochs"]}
        for cls in np.unique(y_probe):
            idx = np.where(y_probe == cls)[0]
            class_label = CLASS_NAME_MAP.get(int(cls), f"class_{int(cls)}")
            class_label = _slugify(class_label)
            for name in ["input_gate", "forget_gate", "candidate_gate", "output_gate", "cell_state", "hidden_state"]:
                flat_means[f"{class_label}__{name}"] = export[name][:, idx].mean(axis=1).astype(np.float32)
        np.savez_compressed(mean_dir / f"{stem}__epoch_class_mean_gate_matrices.npz", **flat_means)
        np.savez_compressed(per_task_dir / "epoch_class_mean_gate_matrices.npz", **flat_means)

        manifest = {
            "task": run["task"],
            "label_mode": run["label_mode"],
            "probe_split": run.get("epoch_gate_probe", {}).get("split", EPOCH_GATE_PROBE_SPLIT),
            "probe_windows": int(export["X_probe"].shape[0]),
            "epochs": export["epochs"].tolist(),
            "X_probe_shape": list(export["X_probe"].shape),
            "epoch_gate_shape": list(export["input_gate"].shape),
            "epoch_gate_shape_definition": "(epochs, probe_windows, time_samples, hidden_size)",
            "gate_keys": GATE_MATRIX_KEYS,
            "notes": {
                "input_gate": "I_t, sigmoid output, range 0 to 1",
                "forget_gate": "F_t, sigmoid output, range 0 to 1",
                "candidate_gate": "c_tilde_t, tanh candidate update, range -1 to 1",
                "output_gate": "O_t, sigmoid output, range 0 to 1",
                "cell_state": "C_t, accumulated memory state",
                "hidden_state": "H_t, exposed/output hidden state",
                "fixed_probe_set": "The same probe windows are used after every epoch so animations reflect training changes rather than changing inputs.",
            },
        }
        _save_json(manifest, epoch_dir / f"{stem}__epoch_gate_matrices_manifest.json")
        _save_json(manifest, per_task_dir / "epoch_gate_matrices_manifest.json")
        saved_any = True

    if saved_any:
        print(f"Saved epoch-wise gate matrix exports to {epoch_dir}")
    else:
        print("[WARN] No epoch-wise gate matrix exports were saved.")


def run_single_regimen(condition, rhythm, results_root=RESULTS_ROOT):
    condition_name = condition["condition_name"]
    rhythm_regime_name = rhythm["rhythm_regime_name"]
    run_name = f"{condition_name}__{rhythm_regime_name}"
    run_dir = results_root / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    cfg = build_cfg(condition, rhythm)
    tasks = build_tasks(cfg, condition_name, rhythm_regime_name)

    metadata = {
        "seed": SEED,
        "device": str(DEVICE),
        "condition": condition,
        "rhythm_regime": rhythm,
        "cfg": cfg.__dict__,
        "tasks": tasks,
        "label_modes": LABEL_MODES,
        "epoch_gate_export": {
            "enabled": RUN_EPOCH_GATE_MATRIX_EXPORT,
            "probe_split": EPOCH_GATE_PROBE_SPLIT,
            "max_probe_windows_per_class": MAX_EPOCH_GATE_PROBE_WINDOWS_PER_CLASS,
        },
    }
    _save_json(metadata, run_dir / "metadata.json")

    print("=" * 80)
    print(f"Running regimen: {run_name}")
    print(f"Saving to: {run_dir}")
    print(cfg)

    set_global_seed(SEED)

    split_info = split_labeled_beta_data(cfg=cfg, n_channels=1)
    _save_json(
        {
            "source_path": split_info["source_path"],
            "train_fraction": split_info["train_fraction"],
            "split_sample": split_info["split_sample"],
            "n_samples": split_info["n_samples"],
            "label_convention": split_info["label_convention"],
            "feature_description": "FFT log-band-power per sliding window",
            "bandpower_hz": cfg.bandpower_hz,
        },
        run_dir / "loaded_data_split.json",
    )

    if RUN_CONTINUOUS_PREVIEWS:
        preview_dir = run_dir / "continuous_previews"
        for task_name, active_frequency_hz, idle_frequency_hz in tasks:
            _capture_and_save_figures(
                preview_task,
                preview_dir,
                _simple_task_name(task_name),
                task_name=task_name,
                active_amp=cfg.active_amps,
                active_frequency_hz=active_frequency_hz,
                idle_frequency_hz=idle_frequency_hz,
                fs=cfg.train_fs,
                duration_sec=PREVIEW_DURATION_SEC,
                idle_amp=cfg.idle_amp,
                seed=SEED,
                signal_noise_std=cfg.signal_noise_std,
                feature_names=FEATURE_NAMES_1CH,
                split='train',
            )

    prepared_runs = prepare_window_experiments(
        cfg=cfg,
        tasks=tasks,
        label_modes=LABEL_MODES,
        seed=SEED,
    )

    prepared_rows = []
    for key, prepared in prepared_runs.items():
        prepared_rows.append({
            "task": prepared["task"],
            "label_mode": prepared["label_mode"],
            "n_train_windows": len(prepared["y_train"]),
            "n_test_windows": len(prepared["y_test"]),
            "window_shape_train": list(prepared["X_train"].shape),
            "window_shape_test": list(prepared["X_test"].shape),
        })
    pd.DataFrame(prepared_rows).to_csv(run_dir / "prepared_window_summary.csv", index=False)

    if RUN_WINDOW_EXAMPLE_PLOTS:
        _capture_and_save_figures(
            plot_all_window_examples,
            run_dir / "window_examples",
            "window_examples",
            prepared_runs,
            n_windows=WINDOW_EXAMPLE_N_WINDOWS,
            split=WINDOW_EXAMPLE_SPLIT,
            feature_names=FEATURE_NAMES_1CH,
        )

    runs, results_df = train_prepared_window_experiments(
        prepared_runs=prepared_runs,
        cfg=cfg,
        seed=SEED,
    )
    save_run_artifacts(runs, results_df, run_dir)

    if RUN_GATE_MATRIX_EXPORT:
        save_gate_matrix_artifacts(
            runs=runs,
            run_dir=run_dir,
            max_windows=MAX_WINDOWS_FOR_GATE_MATRIX_EXPORT,
        )

    if RUN_EPOCH_GATE_MATRIX_EXPORT:
        save_epoch_gate_matrix_artifacts(
            runs=runs,
            run_dir=run_dir,
        )

    if PLOT_ACCURACY_AND_TRAINING:
        _capture_and_save_figures(
            plot_accuracy_and_training_curves,
            run_dir / "accuracy_and_training",
            "accuracy_and_training",
            results_df, runs, tasks, LABEL_MODES,
        )

    if PLOT_PREDICTION_TRACES:
        _capture_and_save_figures(
            plot_all_prediction_traces,
            run_dir / "prediction_traces",
            "prediction_trace",
            runs,
            max_windows=PREDICTION_TRACE_MAX_WINDOWS,
        )

    if RUN_LSTM_DIAGNOSTICS:
        diagnostics_dir = run_dir / "lstm_diagnostics"
        diagnostics_dir.mkdir(parents=True, exist_ok=True)
        diagnostics = run_lstm_diagnostics(
            runs=runs,
            tasks=tasks,
            label_modes=LABEL_MODES,
            feature_names=FEATURE_NAMES_1CH,
            class_name_map=CLASS_NAME_MAP,
            max_windows_per_class=MAX_WINDOWS_PER_CLASS_FOR_DIAGNOSTICS,
            show_input_weight_summary=SHOW_INPUT_WEIGHT_SUMMARY,
            show_channel_contribution_summary=SHOW_CHANNEL_CONTRIBUTION_SUMMARY,
            show_gate_state_summary=SHOW_GATE_STATE_SUMMARY,
        )
        for diagnostic_name, diagnostic_df in diagnostics.items():
            if diagnostic_df is not None and len(diagnostic_df) > 0:
                diagnostic_df.to_csv(diagnostics_dir / f"{diagnostic_name}.csv", index=False)


    if RUN_INTERNAL_CHANNEL_USE_SUMMARY:
        internal_channel_use_df = _capture_and_save_figures(
            run_internal_channel_use_summary,
            run_dir / "internal_channel_use_summary",
            "internal_channel_use",
            runs=runs,
            tasks=tasks,
            label_modes=LABEL_MODES,
            feature_names=FEATURE_NAMES_1CH,
            class_name_map=CLASS_NAME_MAP,
            max_windows_per_class=MAX_WINDOWS_PER_CLASS_FOR_INTERNAL_SUMMARY,
            plot_state_trajectories=PLOT_INTERNAL_STATE_TRAJECTORIES,
        )

        if internal_channel_use_df is not None and len(internal_channel_use_df) > 0:
            internal_channel_use_df.to_csv(
                run_dir / "internal_channel_use_summary" / "internal_channel_use_summary.csv",
                index=False,
            )

    return {
        "run_name": run_name,
        "run_dir": str(run_dir),
        "cfg": cfg,
        "tasks": tasks,
        "results_df": results_df,
    }

def run_all_regimens(condition_regimens=CONDITION_REGIMENS, rhythm_regimens=RHYTHM_REGIMENS, results_root=RESULTS_ROOT):
    summaries = []
    for condition in condition_regimens:
        for rhythm in rhythm_regimens:
            out = run_single_regimen(condition, rhythm, results_root=results_root)
            df = out["results_df"].copy()
            df["condition_name"] = condition["condition_name"]
            df["rhythm_regime_name"] = rhythm["rhythm_regime_name"]
            summaries.append(df)

    all_results_df = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    all_results_df.to_csv(results_root / "all_regimen_results.csv", index=False)
    return all_results_df



In [35]:

# =============================================================================
# Execute all regimens
# =============================================================================
all_regimen_results_df = run_all_regimens()
display(all_regimen_results_df)


Running regimen: loaded_cont_data__loaded_idle_vs_beta
Saving to: /content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold/loaded_cont_data__loaded_idle_vs_beta
Config(train_fs=200, test_fs=200, train_fraction=0.75, window_sec=1.0, stride_sec=0.25, bandpower_hz=(18.0, 22.0), idle_amp=(1.0,), active_amps=(1.0,), signal_noise_std=(0.0,), idle_beta_freq_hz=(20.0,), beta_freq_hz=(20.0,), batch_size=64, epochs=12, lr=0.001, hidden_size=4, num_layers=1)
Using existing labeled file: /content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/cont_data_labeled.npz
Loaded 120000 samples from /content/drive/MyDrive/1 - BME PhD/BCI Lab/online classification/signal_generator/cont_data_labeled.npz at fs=200 Hz
Label counts: {np.int64(0): np.int64(60800), np.int64(1): np.int64(59200)}
Saved 1 figure(s) to /content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold/loaded_cont_data__loaded_idle_vs_beta/continuo

label_mode,endpoint
task,
loaded_cont_data | loaded_idle_vs_beta | Idle vs Beta,0.973199


Saved 2 figure(s) to /content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold/loaded_cont_data__loaded_idle_vs_beta/accuracy_and_training
Saved 1 figure(s) to /content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold/loaded_cont_data__loaded_idle_vs_beta/prediction_traces

=== loaded_cont_data | loaded_idle_vs_beta | Idle vs Beta (endpoint) | three-channel LSTM diagnostics ===
Input-to-gate weight summary:


input_channel,channel 1
gate,
candidate_gate,0.246443
forget_gate,0.230100
input_gate,0.678884
output_gate,0.349748


Channel-resolved gate contribution summary:


...

,class,class_name,quantity,mean,mean_abs,std,min,max,task,label_mode
0,0,Idle,input_gate_pre,0.442577,1.209943,1.243347,-0.808797,2.055241,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
1,0,Idle,forget_gate_pre,0.363464,0.363464,0.226637,0.138212,0.738450,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
2,0,Idle,candidate_gate_pre,-0.192530,0.685926,0.726828,-1.057047,0.782660,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
3,0,Idle,output_gate_pre,0.402407,0.867759,0.909674,-0.656381,1.605595,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
4,0,Idle,input_gate,0.574142,0.574142,0.259429,0.308668,0.885074,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
5,0,Idle,forget_gate,0.588547,0.588547,0.053389,0.534505,0.676487,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
6,0,Idle,candidate_gate,-0.131742,0.559380,0.585419,-0.782676,0.653997,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
7,0,Idle,output_gate,0.581051,0.581051,0.200871,0.341955,0.832705,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
8,0,Idle,cell_state,-0.216797,0.354654,0.359727,-0.609001,0.214268,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
9,0,Idle,hidden_state,-0.173950,0.223907,0.224580,-0.405364,0.073355,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint



=== loaded_cont_data | loaded_idle_vs_beta | Idle vs Beta (endpoint) | compact internal channel-use summary ===


,class,class_name,gate,input_channel,mean_abs_contribution,within_gate_fraction,task,label_mode
0,0,Idle,input_gate,channel 1,0.646355,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
1,0,Idle,forget_gate,channel 1,0.219074,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
2,0,Idle,candidate_gate,channel 1,0.234634,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
3,0,Idle,output_gate,channel 1,0.332989,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
4,1,Beta,input_gate,channel 1,0.702257,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
5,1,Beta,forget_gate,channel 1,0.238022,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
6,1,Beta,candidate_gate,channel 1,0.254927,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint
7,1,Beta,output_gate,channel 1,0.361789,1.0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint


Saved 2 figure(s) to /content/drive/MyDrive/BCI/online classification/signal_generator/sg_results_fixed_threshold/loaded_cont_data__loaded_idle_vs_beta/internal_channel_use_summary


,task,label_mode,bandpower_hz,feature_description,train_windows,test_windows,window_shape_train,window_shape_test,test_acc,condition_name,rhythm_regime_name
0,loaded_cont_data | loaded_idle_vs_beta | Idle ...,endpoint,"(18.0, 22.0)",single FFT log-band-power feature per sliding ...,1797,597,"[1797, 1, 1]","[597, 1, 1]",0.973199,loaded_cont_data,loaded_idle_vs_beta


Each regimen is saved to its own subfolder under the configured results directory.

For each regimen, the notebook writes:

- `metadata.json`
- `loaded_data_split.json`
- `prepared_window_summary.csv`
- `results_summary.csv`
- `prediction_tables/*.csv`
- `training_histories/*.csv`
- `model_state_dicts/*.pt`
- saved figures for:
  - continuous previews
  - window examples
  - accuracy/training curves
  - prediction traces
  - internal hidden/cell trajectories
  - gate/state matrix exports, if enabled
